# Stadt Zürich — Event-Kalender 2023–2025

Strukturierter Event-Kalender als Feature für die Tram-Verspätungsanalyse.

| Kategorie | Quelle | Einträge (2023–2025) |
| :--- | :--- | :--- |
| **Feiertage** | Python `holidays`-Package | 36 |
| **Stadtfeste** | Gemini | 12 |
| **Konzerte** | Perplexity | 5 |
| **Fachmessen & Kongresse** | Perplexity | 83 |
| **Fussball** | Transfermarkt.de | ~115 |

**Output:** `data/interim/events/events-master.csv`

# Dokumentation: Event-Kalender für Tram-IST-Analyse

---

## Zielsetzung

Aufbau eines strukturierten Event-Kalenders für Zürich (2023–2025), der als Feature für die Verspätungsanalyse und das ML-Modell genutzt wird.

---

## Gewichtungsschema

**Schwellenwert:** Nur Events mit voraussichtlich >1.000 Besuchern werden berücksichtigt — kleinere Veranstaltungen haben keinen messbaren Einfluss auf das Tramnetz.

| Gewichtung | Bedeutung | Besucher (real) | Beispiele |
| :--- | :--- | :--- | :--- |
| `3` — Sehr hoch | Massen-Event, stadtweite Auswirkung | >30.000 | Street Parade (~1 Mio.), Züri Fäscht (~2 Mio. über 3 Tage), Taylor Swift (~40.000/Abend), AC/DC (~40.000) |
| `2` — Hoch | Grosser Event, lokaler Impact | 10.000–30.000 | FCZ Super League (~11.500 real), Silvesterzauber (~50.000), Auto Zürich (~10.000/Tag), Ed Sheeran (~35.000) |
| `1` — Mittel | Mittlerer Event | 1.000–10.000 | GCZ Super League (~5.000–8.000 real), Schweizer Cup, Kongresse, Fachmessen, Feiertage |

> **Hinweis Fussball:** Die offiziellen Zuschauerzahlen der Clubs weichen stark von den realen ab. FCZ meldet im Schnitt ~15.000, tatsächlich kommen ~11.500. Ursache: Dauerkarteninhaber werden mitgezählt, erscheinen aber nicht immer. Quelle: Tages-Anzeiger, September 2025.

---

## Datenquellen

Verweis auf vbz-data

---

## Lokale Dateipfade

Verweis auf vbz-data


---

## Datenwörterbucher



In [1]:
from zh_tram_data.doc_loader import show_doc

In [2]:
show_doc('event_subs')

Fehler: Datei nicht gefunden unter /Users/kaywiegand/Workspace/zh-tram-data/data/raw/events/event-subs.md
ROOT_DIR wurde erkannt als: /Users/kaywiegand/Workspace/zh-tram-data


In [3]:

import pandas as pd
from pathlib import Path

# ROOT automatisch finden — egal wo das Notebook liegt
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / 'data' / 'raw').exists():
        ROOT = _p
        break

PATH = str(ROOT / 'data' / 'raw' / 'events') + '/'
print(f'METEO_PATH: {PATH}')



METEO_PATH: /Users/kaywiegand/Workspace/zh-tram-data/data/raw/events/


---

## Feiertage

Gesetzliche Feiertage des Kantons Zürich über das Python-Package `holidays` automatisch generiert.  
Nicht-gesetzliche Feiertage (z.B. Berchtoldstag, Sechseläuten, Knabenschiessen) wurden über Gemini recherchiert und ergänzt.

In [4]:
# Laden der Feiertage
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_holidays = pd.read_csv(PATH + 'holidays.csv', sep=';')
df_holidays['Datum'] = pd.to_datetime(df_holidays['Datum'])

df_holidays.info()
df_holidays.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       36 non-null     datetime64[ns]
 1   Event_Name  36 non-null     object        
 2   Typ         36 non-null     object        
 3   Gewichtung  36 non-null     float64       
 4   Ort         36 non-null     object        
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 1.5+ KB


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-01,Neujahrstag,Feiertag,1.0,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1.0,Zürich
2,2023-04-07,Karfreitag,Feiertag,1.0,Zürich
3,2023-04-10,Ostermontag,Feiertag,1.0,Zürich
4,2023-04-17,Sechseläuten,Feiertag,1.0,Zürich


---

## Stadtfeste

Grosse Zürcher Stadtfeste mit >1.000 Besuchern, recherchiert über Gemini.  
Erfasste Events: **Züri Fäscht**, **Street Parade**, **Silvesterzauber**.

In [5]:
# Laden der Stadtfeste
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_parties = pd.read_csv(PATH + 'parties.csv', sep=';')
df_parties['Datum'] = pd.to_datetime(df_parties['Datum'])

df_parties.info()
df_parties.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       12 non-null     datetime64[ns]
 1   Event_Name  12 non-null     object        
 2   Typ         12 non-null     object        
 3   Gewichtung  12 non-null     int64         
 4   Ort         12 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 608.0+ bytes


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-07-07,Züri Fäscht,Stadtfest,3,Zürich
1,2023-07-08,Züri Fäscht,Stadtfest,3,Zürich
2,2023-07-09,Züri Fäscht,Stadtfest,3,Zürich
3,2023-08-12,Street Parade,Stadtfest,3,Zürich
4,2023-12-31,Silvesterzauber,Stadtfest,2,Zürich


---

## Konzerte

Grosskonzerte im Stadion Letzigrund (>1.000 Besucher), recherchiert über Perplexity.

In [6]:
# Laden der Konzerte
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_concerts = pd.read_csv(PATH + 'concerts.csv', sep=';')
df_concerts['Datum'] = pd.to_datetime(df_concerts['Datum'])

df_concerts.info()
df_concerts.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       5 non-null      datetime64[ns]
 1   Event_Name  5 non-null      object        
 2   Typ         5 non-null      object        
 3   Gewichtung  5 non-null      int64         
 4   Ort         5 non-null      object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 328.0+ bytes


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2024-06-29,AC/DC,Konzert,3,Stadion Letzigrund
1,2024-07-09,Taylor Swift,Konzert,3,Stadion Letzigrund
2,2024-07-10,Taylor Swift,Konzert,3,Stadion Letzigrund
3,2025-08-02,Ed Sheeran,Konzert,2,Stadion Letzigrund
4,2025-08-03,Ed Sheeran,Konzert,2,Stadion Letzigrund


---

## Fachmessen & Kongresse

Grosse Messen und Kongresse in Zürich, recherchiert über Perplexity.  
Locations: Messe Zürich, Hallenstadion, StageOne Zürich-Oerlikon, Kongresshaus.

In [7]:
# Laden der Fachmessen und Kongresse
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_fairs = pd.read_csv(PATH + 'fairs.csv', sep=';')
df_fairs['Datum'] = pd.to_datetime(df_fairs['Datum'])

df_fairs.info()
df_fairs.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       83 non-null     datetime64[ns]
 1   Event_Name  83 non-null     object        
 2   Typ         83 non-null     object        
 3   Gewichtung  83 non-null     int64         
 4   Ort         83 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 3.4+ KB


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-13,World Crypto Conference,Kongress,1,Zürich
1,2023-01-14,World Crypto Conference,Kongress,1,Zürich
2,2023-01-15,World Crypto Conference,Kongress,1,Zürich
3,2023-03-28,Personal Swiss,Fachmesse,1,Messe Zürich
4,2023-03-29,Personal Swiss,Fachmesse,1,Messe Zürich


---

## Fussball

Heimspiele der Zürcher Fussballclubs, manuell erfasst über Transfermarkt.de.  
Erfasste Vereine: **FC Zürich** (Super League, UEFA), **Grasshopper Club Zürich** (Super League).

> Einzelne Spieltage können durch Verlegungen abweichen und sollten vor der Modellierung gegen die Originaldaten geprüft werden.

In [8]:
# Laden der Fussballspiele
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_soccer = pd.read_csv(PATH + 'soccer.csv', sep=';')
df_soccer['Datum'] = pd.to_datetime(df_soccer['Datum'])

df_soccer.info()
df_soccer.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Datum       165 non-null    datetime64[ns]
 1   Event_Name  165 non-null    object        
 2   Typ         165 non-null    object        
 3   Gewichtung  165 non-null    int64         
 4   Ort         165 non-null    object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 6.6+ KB


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2022-07-23,FCZ vs FC Luzern,Super League,2,Letzigrund
1,2022-07-24,GCZ vs FC Lugano,Super League,1,Letzigrund
2,2022-07-27,FCZ vs Qarabağ Ağdam,UEFA Champions League Qual.,2,Letzigrund
3,2022-08-06,GCZ vs FC St. Gallen 1879,Super League,1,Letzigrund
4,2022-08-07,FCZ vs FC Sion,Super League,2,Letzigrund


---

## Master-Kalender

Alle Kategorien zusammenführen, auf 2023–2025 einschränken und nach Datum sortieren.

In [9]:
# Alle Quellen zusammenführen
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
sources = [df_holidays, df_parties, df_concerts, df_fairs, df_soccer]
df_events = pd.concat(sources, ignore_index=True)
df_events['Datum'] = pd.to_datetime(df_events['Datum'])

# Filterung auf Periode 2023–2025
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_events = (
    df_events[
        (df_events['Datum'] >= '2023-01-01') &
        (df_events['Datum'] <= '2025-12-31')
    ]
    .sort_values('Datum')
    .reset_index(drop=True)
)

# Überblick
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print(df_events.groupby(['Typ', 'Gewichtung']).size().rename('Anzahl').to_string())
display(df_events.head(10))

Typ                           Gewichtung
Fachmesse                     1.0           20
                              2.0           34
Feiertag                      1.0           36
Kongress                      1.0           29
Konzert                       2.0            2
                              3.0            3
Schweizer Cup                 1.0            4
Stadtfest                     2.0            5
                              3.0            6
Super League                  1.0           57
                              2.0           58
Super League Barrage          1.0            2
UEFA Conference League Qual.  2.0            2


,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-01,Neujahrstag,Feiertag,1.0,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1.0,Zürich
2,2023-01-13,World Crypto Conference,Kongress,1.0,Zürich
3,2023-01-14,World Crypto Conference,Kongress,1.0,Zürich
4,2023-01-15,World Crypto Conference,Kongress,1.0,Zürich
5,2023-01-21,GCZ vs BSC Young Boys,Super League,1.0,Letzigrund
6,2023-01-29,FCZ vs FC St. Gallen 1879,Super League,2.0,Letzigrund
7,2023-02-01,GCZ vs FC Basel 1893,Schweizer Cup,1.0,Letzigrund
8,2023-02-04,GCZ vs FC Basel 1893,Super League,1.0,Letzigrund
9,2023-02-12,FCZ vs FC Winterthur,Super League,2.0,Letzigrund


---

## Datenwörterbuch — Master-Kalender

| Spalte | Dtype | Beschreibung |
| :--- | :--- | :--- |
| **Datum** | datetime64[us] | Datum des Events (tagesgenau, keine Uhrzeit). Join-Schlüssel für Tram-IST-Daten über `dt.normalize()`. |
| **Event_Name** | str | Bezeichnung des Events. |
| **Typ** | str | Kategorie: Feiertag, Stadtfest, Konzert, Fachmesse, Kongress, Super League, Schweizer Cup, UEFA … |
| **Gewichtung** | float64 | Besucherintensität: 1 = Mittel (1k–10k), 2 = Hoch (10k–30k), 3 = Sehr hoch (>30k). |
| **Ort** | str | Veranstaltungsort / Stadtteil. |

> **EDA-Aufgaben:** Binäres Feature `has_event` (0/1), Kombination von Gewichtung und Typ als kategoriale Variable, Analyse Verspätung vs. Gewichtungsstufe.

In [10]:
from zh_tram_data.doc_loader import show_doc

show_doc('events')

Fehler: Datei nicht gefunden unter /Users/kaywiegand/Workspace/zh-tram-data/data/interim/events/events-master.md
ROOT_DIR wurde erkannt als: /Users/kaywiegand/Workspace/zh-tram-data


---

## Export

Ausgabe als CSV — klein genug für tabellarische Ansicht, menschenlesbar für manuelle Korrekturen.

In [11]:
# Ausgabe-Verzeichnis anlegen (falls nicht vorhanden)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Path('../data/interim/events/').mkdir(parents=True, exist_ok=True)

# Export
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
df_events.to_csv('../data/interim/events/events-master.csv', index=False, sep=';')

print(f'Exportiert: {len(df_events):,} Einträge')
print(f'Zeitraum:   {df_events["Datum"].min().date()} bis {df_events["Datum"].max().date()}')
print(f'Kategorien: {df_events["Typ"].nunique()} Typen, {df_events["Gewichtung"].nunique()} Gewichtungsstufen')

Exportiert: 258 Einträge
Zeitraum:   2023-01-01 bis 2025-12-31
Kategorien: 9 Typen, 3 Gewichtungsstufen


In [12]:
import pandas as pd
df = pd.read_csv('../data/interim/events/events-master.csv', sep=';')
display(df.head())

,Datum,Event_Name,Typ,Gewichtung,Ort
0,2023-01-01,Neujahrstag,Feiertag,1.0,Zürich
1,2023-01-02,Berchtoldstag,Feiertag,1.0,Zürich
2,2023-01-13,World Crypto Conference,Kongress,1.0,Zürich
3,2023-01-14,World Crypto Conference,Kongress,1.0,Zürich
4,2023-01-15,World Crypto Conference,Kongress,1.0,Zürich
